In [1]:
import datetime as dt
from collections import Counter

import optuna

import luigi
from src.auction_model import TargetType, AuctionModelCalibration, RecalibrationMode, AuctionModelInference
import pandas as pd
from config import LUIGI_OUTPUT_DIR as OUT_DIR
from src.tasks import AuctionDates


luigi.interface.core.log_level = "WARNING"
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
luigi.build([AuctionDates(out_dir=OUT_DIR)], local_scheduler=True)
auction_dates = AuctionDates(out_dir=OUT_DIR).read_output().query('20240101 < announce_date < 20260225')['announce_date'].dt.date.to_list()
auction_dates[0], auction_dates[-1]

(datetime.date(2024, 1, 16), datetime.date(2026, 2, 24))

In [4]:
options = [('QuerySoftMax', 'QuerySoftMax'),
 ('QuerySoftMax', 'NDCG:top=5;type=Base'),
 ('YetiRank', 'QuerySoftMax'),
 ('YetiRank', 'NDCG:top=5;type=Base'),
 ('YetiRank:mode=NDCG', 'QuerySoftMax'),
 ('YetiRank:mode=NDCG', 'NDCG:top=5;type=Base'),
 ('Logloss', 'Logloss')]

options = [('QuerySoftMax', 'QuerySoftMax', (), True),
           # ('QuerySoftMax', 'QuerySoftMax', (), False),
           # ('QuerySoftMax', 'QuerySoftMax', ('remaining_size_mln', 'nb_auctions_since_prev1_placement', 'SHORT_TYPE'), False),
           # ('Logloss', 'Logloss', (), True),
           # ('Logloss', 'Logloss', (), False),
           ]

inference_tasks = []
for loss_function, eval_metric, features_override, regime_switch in options:
    for target_type in [TargetType.FIRST]:
        kwargs = dict(time_of_day=dt.timedelta(hours=23), out_dir=OUT_DIR, target_type=target_type,
                      recalibration_mode=RecalibrationMode.QUARTERLY, features_override=features_override, regime_switch=regime_switch)
        extra_tasks = [AuctionModelInference(date=date, loss_function=loss_function, eval_metric=eval_metric, **kwargs).clone_previous() for date in auction_dates]
        inference_tasks += extra_tasks
counter = Counter(x.complete() for x in set(t.requires()[AuctionModelCalibration] for t in inference_tasks))
print(f'calibrations count: {counter}')

calibrations count: Counter({True: 9})


In [5]:
luigi.build(inference_tasks, local_scheduler=True)

True

In [7]:
TOP_K = 3
precision_dfs = {}
raw_dfs = {}

for target_type in [TargetType.FIRST]:
    for loss_function, eval_metric, features_override, regime_switch in options:
        kwargs = dict(time_of_day=dt.timedelta(hours=23), out_dir=OUT_DIR, target_type=target_type, recalibration_mode=RecalibrationMode.QUARTERLY, features_override=features_override, regime_switch=regime_switch)
        tasks = [AuctionModelInference(date=date, loss_function=loss_function, eval_metric=eval_metric, **kwargs).clone_previous() for date in auction_dates]
        df = pd.concat([t.read_output().reset_index() for t in tasks], ignore_index=True)
        df['time_bucket'] = df['prediction_ts'].dt.to_period('Q').map(lambda p: p.start_time.date())
        df = df.sort_values(['target_date', 'score', 'years_to_maturity'], ascending=[True, False, False])
        df['proba_proxy'] = df['proba'] / (df.groupby('target_date')['proba'].transform('sum')) * 2
        model_code = loss_function + '_' + eval_metric + (f'_reduced_{len(features_override)}' if len(features_override) > 0 else '') + ('_regime_switch' if regime_switch else '')
        raw_dfs.setdefault(target_type, {})[model_code] = df
        precision_dfs.setdefault(target_type, {})[model_code] = \
        df.dropna(subset=['target']).groupby('target_date').head(TOP_K).groupby('time_bucket')['target'].mean() * TOP_K

    for baseline_cutoff in [10, 100]:
        mask = (df['remaining_size_mln'] >= baseline_cutoff * 1000) | (df.groupby('target_date')['remaining_size_mln'].rank(ascending=False, method='first') <= TOP_K)
        baseline_df = df[mask].dropna(subset=['target']).groupby(['target_date', 'time_bucket'], as_index=False)['target'].agg(['sum', 'count'])
        assert all(baseline_df['count'] >= TOP_K)
        baseline_df['target'] = baseline_df['sum'] / baseline_df['count'] * TOP_K
        precision_dfs.setdefault(target_type, {})[f'baseline_{baseline_cutoff}'] = baseline_df.groupby('time_bucket')['target'].mean()


In [8]:
pd.concat(precision_dfs[TargetType.FIRST]).unstack().T.plot(backend='plotly', kind='bar', barmode='group').show()

In [9]:
mdf = next(iter(raw_dfs[TargetType.FIRST].values())).query('target == 1').groupby('target_date')['years_to_maturity'].agg(['max', 'min'])
mdf['10Y'] = 10
# mdf['delta'] = mdf['max'] - mdf['min']
mdf.plot(backend='plotly')